# XModalix3D
## Full run report

This notebook is the XModalix3D counterpart of the `imagix3d_full_report_template.ipynb`.

It is intended to provide a reproducible report of a trained **cross-modal 3D VAE** run, with particular emphasis on:

- the modality-specific sub-VAEs and their optimization settings,
- XModalix-specific loss components,
- within-modality 3D reconstruction quality,
- cross-modal translation quality,
- latent-space structure,
- quantitative cross-modal alignment using **modality linear probing** and **matched/unmatched patient distances**,
- optional downstream embedding evaluation.

The report deliberately distinguishes two aspects of cross-modal alignment:

1. **Modality invariance**: whether modality identity can still be linearly decoded from the latent representation.
2. **Patient-specific correspondence**: whether embeddings of the same patient are closer across modalities than embeddings of different patients.

For model-development decisions, use the **validation split**. The **test split should be retained for final reporting rather than model selection**.


In [ ]:
# Parameters

# These values can either be filled in here or supplied as environment variables
# by the report-generating job.
RESULT_DIR = ""
PKL_PATH = ""
REPORT_DIR = ""

# Optional downstream embedding evaluation.
EVALUATION_PARAMS = ["median_split"]

# Report controls
PRINT_FULL_MODELS = True
N_RECON_SAMPLES = 3
N_TRANSLATION_SAMPLES = 3
TRANSLATION_GROUP_PARAM = None
MAX_TRANSLATION_VOXELS = 10000
RECON_METRIC_MAX_SAMPLES = None   # None = all available samples
DIVERSITY_MAX_SAMPLES = 20
MEAN_MAP_MAX_SAMPLES = 30
DPI = 300

ALIGNMENT_REPRESENTATIONS = "z"
LINEAR_PROBE_C = 1.0
LINEAR_PROBE_CV_SPLITS = 5

### Loading the pipeline

The same CPU-safe loading approach as in the Imagix3D report is used here. This is useful when the serialized pipeline contains tensors that were originally created on CUDA.

After loading, the report explicitly replaces the evaluator and visualizer with `XModalix3DEvaluator` and `XModal3DVisualizer` so that the new XModalix3D-specific alignment evaluation and 3D visualization methods are available.


In [ ]:
from pathlib import Path
import io
import json
import os
from enum import Enum

import numpy as np
import pandas as pd
import torch
import torch.distributed as dist
import matplotlib.pyplot as plt
import dill as pickle
import seaborn as sns

import autoencodix as acx

from autoencodix.evaluate._xmodalix3d_evaluator import XModalix3DEvaluator
from autoencodix.visualize._xmodal3d_visualizer import XModal3DVisualizer
from autoencodix.utils._utils import find_translation_keys

RESULT_DIR = os.environ.get("RESULT_DIR", RESULT_DIR)
PKL_PATH = os.environ.get("PKL_PATH", PKL_PATH)
REPORT_DIR = os.environ.get("REPORT_DIR", REPORT_DIR)

if not PKL_PATH:
    raise ValueError("PKL_PATH is empty. Set it above or provide it as an environment variable.")

result_dir = Path(RESULT_DIR).resolve() if RESULT_DIR else Path.cwd().resolve()
pkl_path = Path(PKL_PATH).resolve()
report_dir = Path(REPORT_DIR).resolve() if REPORT_DIR else (result_dir / "xmodalix3d_report")

report_dir.mkdir(parents=True, exist_ok=True)

print("Result directory:", result_dir)
print("Pickle path:", pkl_path)
print("Report directory:", report_dir)

if not pkl_path.is_file():
    raise FileNotFoundError(f"Pickle file does not exist: {pkl_path}")

_original_load_from_bytes = torch.storage._load_from_bytes

def _load_from_bytes_cpu(bytes_object):
    return torch.load(
        io.BytesIO(bytes_object),
        map_location=torch.device("cpu"),
    )

torch.storage._load_from_bytes = _load_from_bytes_cpu

# The saved pipeline contains DDP wrappers.
# Temporarily create a one-process distributed group so that they can be unpickled.
started_dist_here = False

try:
    if not dist.is_initialized():
        os.environ.setdefault("MASTER_ADDR", "127.0.0.1")
        os.environ.setdefault("MASTER_PORT", "29500")

        dist.init_process_group(
            backend="gloo",
            rank=0,
            world_size=1,
        )

        started_dist_here = True

    # Load the pipeline pickle directly.
    with open(pkl_path, "rb") as f:
        xmodalix3d_loaded = pickle.load(f)

    # Remove Fabric/DDP wrappers from the models.
    clean_models = {}

    for model_name, model in xmodalix3d_loaded.result.model.items():

        while hasattr(model, "module"):
            model = model.module

        model = model.cpu()

        # Load the separately saved model state.
        state_path = (
            pkl_path.parent
            / f"{model_name}_{pkl_path.stem}_model.pth"
        )

        state_dict = torch.load(
            state_path,
            map_location="cpu",
        )

        # Older DDP-saved state dicts can contain "module." prefixes.
        while (
            state_dict
            and all(
                key.startswith("module.")
                for key in state_dict
            )
        ):
            state_dict = {
                key[len("module."):]: value
                for key, value in state_dict.items()
            }

        model.load_state_dict(state_dict)

        clean_models[model_name] = model

    xmodalix3d_loaded.result.model = clean_models

    # Restore the separately saved preprocessor.
    preprocessor_path = (
        pkl_path.parent
        / f"{pkl_path.stem}_preprocessor.pkl"
    )

    with open(preprocessor_path, "rb") as f:
        xmodalix3d_loaded._preprocessor = pickle.load(f)

finally:
    torch.storage._load_from_bytes = _original_load_from_bytes

    if started_dist_here and dist.is_initialized():
        dist.destroy_process_group()
        
result = xmodalix3d_loaded.result

print("Loaded XModalix3D pipeline")
print("Evaluator:", type(xmodalix3d_loaded.evaluator).__name__)
print("Visualizer:", type(xmodalix3d_loaded.visualizer).__name__)

In [ ]:
### Validate and, if necessary, reconstruct complete final embeddings

eval_datasets = result.datasets

if (
    eval_datasets is None
    or eval_datasets.train is None
):
    eval_datasets = xmodalix3d_loaded._datasets


representation_refresh = (
    xmodalix3d_loaded.evaluator.refresh_final_representations(
        result=result,
        datasets=eval_datasets,
        batch_size=2,
        device=(
            "cuda:0"
            if torch.cuda.is_available()
            else "cpu"
        ),
        seed=42,
    )
)

display(representation_refresh)

representation_refresh.to_csv(
    report_dir / "final_representation_refresh.csv",
    index=False,
)

In [ ]:
### Verify that refresh preserved translation metadata

# Determine the configured translation direction here so that this
# preflight check does not depend on variables defined later in the report.
if not isinstance(result.model, dict):
    raise TypeError(
        "Expected result.model to be a dictionary of modality-specific models."
    )

trained_modalities = list(result.model.keys())

translation_keys = find_translation_keys(
    config=xmodalix3d_loaded.config,
    trained_modalities=trained_modalities,
)

from_key = translation_keys["from"]
to_key = translation_keys["to"]

print(f"Configured translation: {from_key} -> {to_key}")


test_recons = result.reconstructions.get(
    epoch=-1,
    split="test",
)

test_ids = result.sample_ids.get(
    epoch=-1,
    split="test",
)

if not isinstance(test_recons, dict):
    raise RuntimeError(
        "Final test reconstructions are not stored as a dictionary."
    )

if not isinstance(test_ids, dict):
    raise RuntimeError(
        "Final test sample IDs are not stored as a dictionary."
    )


reference_key = f"reference_{to_key}_to_{to_key}"

print(
    "Test reconstruction keys:",
    list(test_recons.keys()),
)

print(
    "Test sample-ID keys:",
    list(test_ids.keys()),
)


for key in ["translation", reference_key]:

    if key not in test_recons:
        raise RuntimeError(
            f"Missing reconstruction after refresh: {key}"
        )

    if key not in test_ids:
        raise RuntimeError(
            f"Missing reconstruction sample IDs after refresh: {key}"
        )

    n_recons = len(test_recons[key])
    n_ids = len(test_ids[key])

    if n_recons != n_ids:
        raise RuntimeError(
            f"{key}: {n_recons} reconstructions but "
            f"{n_ids} sample IDs."
        )

    print(
        f"{key}: "
        f"{n_recons} reconstructions / "
        f"{n_ids} IDs: OK"
    )


print(
    "\nTranslation metadata survived representation refresh."
)

### Helper functions for stored epochs and array conversion

In [ ]:
def _nonempty(value):
    if value is None:
        return False
    if isinstance(value, dict):
        return len(value) > 0
    try:
        return len(value) > 0
    except TypeError:
        return True


def stored_epochs_for_split(store, split):
    epochs = []
    data = getattr(store, "_data", {})
    if isinstance(data, dict):
        for epoch, split_data in data.items():
            if epoch == -1:
                continue
            if isinstance(split_data, dict) and _nonempty(split_data.get(split)):
                epochs.append(epoch)
    return sorted(set(epochs))


def resolve_epoch(store, split):
    # Prefer a final prediction stored at epoch=-1.
    try:
        value = store.get(epoch=-1, split=split)
        if _nonempty(value):
            return -1
    except Exception:
        pass

    epochs = stored_epochs_for_split(store, split)
    if not epochs:
        raise ValueError(
            f"No stored data found for split={split!r} in {type(store).__name__}."
        )
    return max(epochs)


def to_numpy(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().numpy()
    return np.asarray(value)


def make_json_safe(obj):
    if isinstance(obj, Enum):
        return obj.value
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, dict):
        return {str(k): make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [make_json_safe(v) for v in obj]
    return obj


def config_to_dict(config):
    if hasattr(config, "model_dump"):
        cfg = config.model_dump(mode="python")
    elif hasattr(config, "dict"):
        cfg = config.dict()
    else:
        raise TypeError(f"Unsupported config type: {type(config)}")
    return make_json_safe(cfg)


def extract_dataset_array(dataset, index):
    item = dataset[index]

    if isinstance(item, (tuple, list)):
        if len(item) < 2:
            raise ValueError("Dataset item tuple/list does not contain image data.")
        value = item[1]
    else:
        value = item

    return to_numpy(value).squeeze().astype(np.float32)

def resolve_matplotlib_figure(plot_obj, *, label="plot"):
    """Return the underlying matplotlib Figure from common plot containers.

    Supports:
    - matplotlib.figure.Figure
    - AUTOENCODIX Plot-like wrappers exposing .figure
    - seaborn grids exposing .figure or .fig
    - matplotlib Axes exposing .get_figure()

    Raises a descriptive error instead of allowing .savefig() to be called
    accidentally on a wrapper or nested dictionary.
    """
    from collections.abc import Mapping
    from matplotlib.figure import Figure

    current = plot_obj
    seen = set()

    for _ in range(8):
        if isinstance(current, Figure):
            return current

        if isinstance(current, Mapping):
            raise TypeError(
                f"{label}: expected a plot/Figure, but received a mapping "
                f"of type {type(current).__name__}. "
                "This usually means the plot dictionary was indexed with the wrong key."
            )

        object_id = id(current)
        if object_id in seen:
            break
        seen.add(object_id)

        advanced = False

        for attr in ("figure", "fig"):
            if hasattr(current, attr):
                candidate = getattr(current, attr)
                if candidate is not None and candidate is not current:
                    current = candidate
                    advanced = True
                    break

        if advanced:
            continue

        if hasattr(current, "get_figure"):
            candidate = current.get_figure()
            if candidate is not None and candidate is not current:
                current = candidate
                continue

        break

    raise TypeError(
        f"{label}: could not resolve a matplotlib Figure from "
        f"object of type {type(plot_obj).__name__}."
    )


def save_report_plot(
    plot_obj,
    stem,
    *,
    dpi=DPI,
    close=False,
):
    """Save supported plot objects as both PDF and PNG.

    Supports:
    - matplotlib Figure objects
    - AUTOENCODIX / seaborn wrappers exposing .figure or .fig
    - matplotlib Axes
    - seaborn.objects.Plot
    """

    pdf_path = report_dir / f"{stem}.pdf"
    png_path = report_dir / f"{stem}.png"

    # --------------------------------------------------------------
    # seaborn.objects.Plot
    #
    # This is NOT a matplotlib Figure wrapper. It has its own
    # public .save() method and does not expose .figure.
    # --------------------------------------------------------------
    try:
        from seaborn.objects import Plot as SeabornObjectsPlot
    except ImportError:
        SeabornObjectsPlot = None

    if (
        SeabornObjectsPlot is not None
        and isinstance(plot_obj, SeabornObjectsPlot)
    ):
        plot_obj.save(
            pdf_path,
            bbox_inches="tight",
        )

        plot_obj.save(
            png_path,
            dpi=dpi,
            bbox_inches="tight",
        )

        print(
            f"Saved {stem}: "
            f"{pdf_path.name}, {png_path.name} "
            "(seaborn.objects.Plot)"
        )

        # seaborn.objects.Plot.save() compiles its own figure,
        # so there is no persistent Figure handle that needs
        # to be closed here.
        return plot_obj

    # --------------------------------------------------------------
    # Matplotlib figures and objects wrapping matplotlib figures
    # --------------------------------------------------------------
    fig = resolve_matplotlib_figure(
        plot_obj,
        label=stem,
    )

    fig.savefig(
        pdf_path,
        bbox_inches="tight",
    )

    fig.savefig(
        png_path,
        dpi=dpi,
        bbox_inches="tight",
    )

    print(
        f"Saved {stem}: "
        f"{pdf_path.name}, {png_path.name}"
    )

    if close:
        plt.close(fig)

    return fig

### Dataset overview

In [ ]:
import numpy as np
import pandas as pd

dataset = xmodalix3d_loaded.result.datasets

print("\nSample splits (in total):")
for split in ["train", "valid", "test"]:
    split_data = getattr(dataset, split, None)
    if split_data is None:
        print(f"{split}: no split available")
    else:
        print(f"{split}: {len(split_data.sample_ids)} samples")

### Modality-specific VAE models

In [ ]:
models = result.model

if not isinstance(models, dict):
    raise TypeError(
        "Expected XModalix3D result.model to contain a dictionary of modality-specific models."
    )

model_rows = []

for modality, model in models.items():
    module = model.module if hasattr(model, "module") else model

    try:
        n_parameters = int(sum(p.numel() for p in module.parameters()))
        n_trainable = int(sum(p.numel() for p in module.parameters() if p.requires_grad))
    except Exception:
        n_parameters = np.nan
        n_trainable = np.nan

    model_rows.append(
        {
            "modality": modality,
            "model_class": type(module).__name__,
            "n_parameters": n_parameters,
            "n_trainable_parameters": n_trainable,
        }
    )

    print(f"\n{modality}: {type(module).__name__}")
    if PRINT_FULL_MODELS:
        display(module)

model_overview = pd.DataFrame(model_rows)
display(model_overview)

### Configuration overview

In [ ]:
cfg = config_to_dict(xmodalix3d_loaded.config)

print(json.dumps(cfg, indent=2, ensure_ascii=False))

with open(report_dir / "config.json", "w", encoding="utf-8") as handle:
    json.dump(cfg, handle, indent=2, ensure_ascii=False)


### Translation direction and stored XModalix3D outputs

In [ ]:
trained_modalities = list(models.keys())

translation_keys = find_translation_keys(
    config=xmodalix3d_loaded.config,
    trained_modalities=trained_modalities,
)

from_key = translation_keys["from"]
to_key = translation_keys["to"]

print(f"Configured translation: {from_key} -> {to_key}")

for split in ["train", "valid", "test"]:
    try:
        epoch = resolve_epoch(result.reconstructions, split)
        reconstruction_dict = result.reconstructions.get(
            epoch=epoch,
            split=split,
        )
        print(
            f"{split}: reconstruction epoch={epoch}, "
            f"keys={list(reconstruction_dict.keys()) if isinstance(reconstruction_dict, dict) else type(reconstruction_dict)}"
        )
    except Exception as exc:
        print(f"{split}: no reconstruction output available ({exc})")

print("\nAvailable sub-loss terms:")
try:
    print(list(result.sub_losses.keys()))
except Exception as exc:
    print("Could not list sub-loss terms:", exc)


### Latent representations

The following cell exports the sampled latent vectors $z$ for each available split and modality.


In [ ]:
latent_summary_rows = []

representation_stores = {
    "z": result.latentspaces,
}

for split in ["train", "valid", "test"]:
    for representation_name, store in representation_stores.items():
        try:
            epoch = resolve_epoch(store, split)
            values_by_modality = store.get(epoch=epoch, split=split)
        except Exception as exc:
            print(f"Skipping {representation_name}, {split}: {exc}")
            continue

        if not isinstance(values_by_modality, dict):
            continue

        try:
            sample_ids_by_modality = result.sample_ids.get(
                epoch=epoch,
                split=split,
            )
        except Exception:
            sample_ids_by_modality = {}

        for modality, values in values_by_modality.items():
            arr = to_numpy(values)

            if arr.ndim == 1:
                arr = arr.reshape(-1, 1)
            elif arr.ndim > 2:
                arr = arr.reshape(arr.shape[0], -1)

            ids = None
            if isinstance(sample_ids_by_modality, dict):
                ids = sample_ids_by_modality.get(modality)

            frame = pd.DataFrame(
                arr,
                columns=[
                    f"{representation_name}_{i}"
                    for i in range(arr.shape[1])
                ],
            )

            if ids is not None and len(ids) == len(frame):
                frame.insert(0, "sample_id", np.asarray(ids).astype(str))

            safe_modality = modality.replace(".", "_")
            frame.to_csv(
                report_dir
                / f"{representation_name}_{safe_modality}_{split}_epoch_{epoch}.csv",
                index=False,
            )

            numeric = frame.select_dtypes(include=np.number)
            desc = numeric.describe().T

            for latent_dim, row in desc.iterrows():
                latent_summary_rows.append(
                    {
                        "split": split,
                        "epoch": epoch,
                        "representation": representation_name,
                        "modality": modality,
                        "latent_dim": latent_dim,
                        "mean": row.get("mean"),
                        "std": row.get("std"),
                        "min": row.get("min"),
                        "median": row.get("50%"),
                        "max": row.get("max"),
                    }
                )

latent_summary = pd.DataFrame(latent_summary_rows)
display(latent_summary.head(20))

latent_summary.to_csv(
    report_dir / "latent_values_summary.csv",
    index=False,
)

print("Saved modality-specific latent values and summaries to:", report_dir)


In [ ]:
TARGET_MODALITIES = {"NCCT", "CBF"}

tables_by_modality = {
    modality: []
    for modality in TARGET_MODALITIES
}

for split in ["train", "valid", "test"]:

    # Use the final stored/refreshed representation for this split.
    epoch = resolve_epoch(result.latentspaces, split)

    z_by_modality = result.latentspaces.get(
        epoch=epoch,
        split=split,
    )

    sample_ids_by_modality = result.sample_ids.get(
        epoch=epoch,
        split=split,
    )

    if not isinstance(z_by_modality, dict):
        raise TypeError(
            f"Expected latent spaces for split {split!r} "
            "to be stored as a modality dictionary."
        )

    if not isinstance(sample_ids_by_modality, dict):
        raise TypeError(
            f"Expected sample IDs for split {split!r} "
            "to be stored as a modality dictionary."
        )

    for modality_key, values in z_by_modality.items():

        # XModalix keys may be e.g. "NCCT" or "img.NCCT".
        simple_modality = str(modality_key).split(".", 1)[-1].upper()

        if simple_modality not in TARGET_MODALITIES:
            continue

        if modality_key not in sample_ids_by_modality:
            raise KeyError(
                f"No sample IDs found for modality {modality_key!r}, "
                f"split {split!r}."
            )

        z = to_numpy(values)

        if z.ndim == 1:
            z = z.reshape(-1, 1)
        elif z.ndim != 2:
            z = z.reshape(z.shape[0], -1)

        sample_ids = np.asarray(
            sample_ids_by_modality[modality_key]
        ).astype(str)

        if len(sample_ids) != len(z):
            raise ValueError(
                f"{simple_modality}, {split}: "
                f"{len(sample_ids)} sample IDs but "
                f"{len(z)} latent vectors."
            )

        frame = pd.DataFrame(
            z,
            columns=[
                f"LatDim_{i}"
                for i in range(z.shape[1])
            ],
        )

        frame.insert(0, "split", split)
        frame.insert(0, "sample_id", sample_ids)

        tables_by_modality[simple_modality].append(frame)


# ============================================================================
# Combine splits and save one CSV per modality
# ============================================================================

for modality in sorted(TARGET_MODALITIES):

    if not tables_by_modality[modality]:
        raise RuntimeError(
            f"No final z embeddings were found for {modality}."
        )

    z_all = pd.concat(
        tables_by_modality[modality],
        ignore_index=True,
    )

    # ------------------------------------------------------------------------
    # Sanity checks
    # ------------------------------------------------------------------------

    if not z_all["sample_id"].is_unique:
        duplicated = (
            z_all.loc[
                z_all["sample_id"].duplicated(keep=False),
                "sample_id",
            ]
            .unique()
            .tolist()
        )

        raise ValueError(
            f"{modality}: duplicate sample IDs found. "
            f"First duplicates: {duplicated[:10]}"
        )

    latent_columns = [
        column
        for column in z_all.columns
        if column.startswith("LatDim_")
    ]

    latent_values = z_all[latent_columns].to_numpy(dtype=float)

    if not np.isfinite(latent_values).all():
        raise ValueError(
            f"{modality}: non-finite latent values found."
        )

    # ------------------------------------------------------------------------
    # Save
    # ------------------------------------------------------------------------

    output_path = (
        report_dir
        / f"xmodalix3d_{modality.lower()}_final_z_all_splits.csv"
    )

    z_all.to_csv(
        output_path,
        index=False,
    )

    print()
    print(f"{modality}:")
    print(f"  latent dimensionality: {len(latent_columns)}")
    print(f"  total samples: {len(z_all)}")
    print(z_all.groupby("split").size())
    print(f"  saved to: {output_path}")

    display(z_all.head())

### XModalix3D loss overview

Unlike a single VAE, XModalix3D combines multiple loss components:

- modality-specific reconstruction and variational losses,
- adversarial loss,
- paired loss,
- optionally class-based loss,
- the internal modality-classifier loss used during adversarial training.

The table below exports all stored loss curves in long format so that the exact contribution of every loss component remains available.


In [ ]:
def _curve_rows(values, split, loss_term):
    rows = []

    if values is None:
        return rows

    if isinstance(values, pd.Series):
        iterator = values.items()
    elif isinstance(values, dict):
        iterator = values.items()
    else:
        arr = np.asarray(values).squeeze()
        if arr.ndim == 0:
            iterator = [(0, arr)]
        else:
            iterator = enumerate(arr)

    for epoch, value in iterator:
        try:
            scalar = float(np.asarray(value).squeeze())
        except Exception:
            continue

        rows.append(
            {
                "epoch": int(epoch),
                "display_epoch": int(epoch) + 1,
                "split": split,
                "loss_term": loss_term,
                "value": scalar,
            }
        )

    return rows


loss_rows = []

for split in ["train", "valid"]:
    try:
        total_values = result.losses.get(split=split)
        loss_rows.extend(
            _curve_rows(total_values, split, "total_loss")
        )
    except Exception:
        pass

try:
    sub_loss_keys = list(result.sub_losses.keys())
except Exception:
    sub_loss_keys = []

for key in sub_loss_keys:
    try:
        dynamics = result.sub_losses.get(key)
    except Exception:
        continue

    for split in ["train", "valid"]:
        try:
            values = dynamics.get(split=split)
        except Exception:
            continue

        loss_rows.extend(
            _curve_rows(values, split, str(key))
        )

losses_long = pd.DataFrame(loss_rows)

if losses_long.empty:
    print("No stored loss curves found.")
else:
    final_loss_table = (
        losses_long
        .sort_values("epoch")
        .groupby(["split", "loss_term"], as_index=False)
        .tail(1)
        .sort_values(["split", "loss_term"])
        .reset_index(drop=True)
    )

    display(final_loss_table)

    losses_long.to_csv(
        report_dir / "losses_long.csv",
        index=False,
    )
    final_loss_table.to_csv(
        report_dir / "losses_final_values.csv",
        index=False,
    )


### Downstream embedding evaluation

As in the XModalix tutorial, each modality-specific embedding is evaluated separately. This section is disabled by default because it can rerun a complete downstream ML evaluation.

For classification, the example below uses standardized logistic regression and compares the learned embeddings with PCA, UMAP, and randomly selected features.


In [ ]:
from sklearn.linear_model import LogisticRegression

### Evaluate the latent representation

result = xmodalix3d_loaded.result
result.embedding_evaluation = pd.DataFrame()

clf = LogisticRegression(
    solver="lbfgs",
    class_weight="balanced",
    max_iter=2000,
    random_state=42,
)

xmodalix3d_loaded.evaluate(
    params=["median_split"],
    reference_methods=["PCA", "UMAP"],
    reference_reducer={},
    ml_model_class=clf,
    metric_class="roc_auc_ovo",
    split_type="use-split",
    n_downsample=None,
)

embedding_evaluation = (
    result.embedding_evaluation
    .reset_index(drop=True)
)
display(embedding_evaluation)
embedding_evaluation.to_csv(
    report_dir / "embedding_evaluation.csv",
    index=False,
)

# Retrieve the evaluation plot generated by evaluate()
algorithm_plots = (
    xmodalix3d_loaded.visualizer.plots["ML_Evaluation"]
    ["median_split"]
    ["roc_auc_ovo"]
)

# Only one classifier was supplied above.
eval_grid = next(iter(algorithm_plots.values()))

# Save plot through the common figure resolver
save_report_plot(
    eval_grid,
    "latent_space_eval",
    close=True,
)

## 3D reconstruction and translation quality

XModalix3D trains one VAE per modality. We therefore inspect both:

- **within-modality reconstruction** for every modality,
- **cross-modal translation** for the configured source → target direction.

For the target modality, the reference reconstruction is generated from the target modality's own latent representation, while the translation is generated by encoding the source modality and decoding through the target decoder.


### Cross-modal translation grid

In [ ]:
xmodalix3d_loaded.visualizer.show_image_translation(
    result=result,
    from_key=from_key,
    to_key=to_key,
    n_sample_per_class=N_TRANSLATION_SAMPLES,
    param=TRANSLATION_GROUP_PARAM,
    split="test",
)

# show_image_translation() stores ungrouped plots under "all"
translation_plot_key = (
    TRANSLATION_GROUP_PARAM
    if TRANSLATION_GROUP_PARAM is not None
    else "all"
)

fig_translation = (
    xmodalix3d_loaded.visualizer.plots[
        "Image-translation-3D"
    ][to_key]["test"][translation_plot_key]
)

_ = save_report_plot(
    fig_translation,
    "translation_grid_test",
    close=True,
)

### 2D comparison of original and translated target volumes

This is the 3D-compatible counterpart of the XModalix tutorial's `show_2D_translation()`. Complete 3D volumes are too large to flatten directly, so `XModal3DVisualizer` uniformly subsamples voxel positions before applying the dimensionality reduction.


In [ ]:
xmodalix3d_loaded.visualizer.show_2D_translation(
    result=result,
    translated_modality=to_key,
    split="test",
    param=TRANSLATION_GROUP_PARAM,
    reducer="PCA",
    max_voxels=MAX_TRANSLATION_VOXELS,
    include_self_reconstruction=True,
)

# show_2D_translation() stores ungrouped plots under "origin"
translation_2d_plot_key = (
    TRANSLATION_GROUP_PARAM
    if TRANSLATION_GROUP_PARAM is not None
    else "origin"
)

fig_translation_pca = (
    xmodalix3d_loaded.visualizer.plots[
        "2D-translation-3D"
    ][to_key]["test"][translation_2d_plot_key]
)

_ = save_report_plot(
    fig_translation_pca,
    "translation_pca_original_vs_translated",
    close=True,
)

### Quantitative reconstruction and translation metrics per sample

MSE, MAE, RMSE and percentile errors are calculated voxel-wise.

For within-modality reconstructions, each modality is compared with its own VAE reconstruction. For the configured target modality, the report additionally compares the original target volume with both the XModalix3D reference reconstruction and the cross-modal translation.


In [ ]:
def compute_output_metrics(
    result,
    *,
    split,
    modality,
    output_key,
    output_type,
    max_samples=None,
):
    split_data = getattr(result.datasets, split, None)
    if split_data is None or not hasattr(split_data, "datasets"):
        return []

    if modality not in split_data.datasets:
        return []

    dataset = split_data.datasets[modality]

    try:
        epoch = resolve_epoch(result.reconstructions, split)
        output_dict = result.reconstructions.get(
            epoch=epoch,
            split=split,
        )
        id_dict = result.sample_ids.get(
            epoch=epoch,
            split=split,
        )
    except Exception:
        return []

    if not isinstance(output_dict, dict) or output_key not in output_dict:
        return []

    outputs = to_numpy(output_dict[output_key])

    dataset_ids = np.asarray(dataset.sample_ids).astype(str)
    dataset_lookup = {
        sample_id: idx
        for idx, sample_id in enumerate(dataset_ids)
    }

    if isinstance(id_dict, dict) and output_key in id_dict:
        output_ids = np.asarray(id_dict[output_key]).astype(str)
    elif isinstance(id_dict, dict) and modality in id_dict:
        output_ids = np.asarray(id_dict[modality]).astype(str)
    else:
        output_ids = dataset_ids[: len(outputs)]

    n = min(len(outputs), len(output_ids))
    if max_samples is not None:
        n = min(n, int(max_samples))

    rows = []

    for output_idx in range(n):
        sample_id = str(output_ids[output_idx])

        if sample_id not in dataset_lookup:
            continue

        original = extract_dataset_array(
            dataset,
            dataset_lookup[sample_id],
        )
        reconstructed = np.asarray(
            outputs[output_idx]
        ).squeeze().astype(np.float32)

        if original.shape != reconstructed.shape:
            raise ValueError(
                f"Shape mismatch for {sample_id}, {output_key}: "
                f"original={original.shape}, output={reconstructed.shape}"
            )

        abs_error = np.abs(original - reconstructed)
        sq_error = (original - reconstructed) ** 2

        rows.append(
            {
                "split": split,
                "epoch": epoch,
                "modality": modality,
                "output_key": output_key,
                "output_type": output_type,
                "sample_id": sample_id,
                "mse": float(np.mean(sq_error)),
                "mae": float(np.mean(abs_error)),
                "rmse": float(np.sqrt(np.mean(sq_error))),
                "p95_abs_error": float(np.percentile(abs_error, 95)),
                "p99_abs_error": float(np.percentile(abs_error, 99)),
                "max_abs_error": float(np.max(abs_error)),
                "orig_mean": float(np.mean(original)),
                "output_mean": float(np.mean(reconstructed)),
                "orig_std": float(np.std(original)),
                "output_std": float(np.std(reconstructed)),
            }
        )

    return rows


metric_rows = []

# Within-modality reconstructions wherever they are stored.
for split in ["train", "valid", "test"]:
    for modality in trained_modalities:
        metric_rows.extend(
            compute_output_metrics(
                result,
                split=split,
                modality=modality,
                output_key=modality,
                output_type="within_modality_reconstruction",
                max_samples=RECON_METRIC_MAX_SAMPLES,
            )
        )

# Test-set target reference and translation.
try:
    test_epoch = resolve_epoch(result.reconstructions, "test")
    test_recons = result.reconstructions.get(
        epoch=test_epoch,
        split="test",
    )
except Exception:
    test_recons = {}

reference_key = f"reference_{to_key}_to_{to_key}"

if isinstance(test_recons, dict):
    if reference_key in test_recons:
        metric_rows.extend(
            compute_output_metrics(
                result,
                split="test",
                modality=to_key,
                output_key=reference_key,
                output_type="target_reference_reconstruction",
                max_samples=RECON_METRIC_MAX_SAMPLES,
            )
        )

    if "translation" in test_recons:
        metric_rows.extend(
            compute_output_metrics(
                result,
                split="test",
                modality=to_key,
                output_key="translation",
                output_type="cross_modal_translation",
                max_samples=RECON_METRIC_MAX_SAMPLES,
            )
        )

reconstruction_metrics = pd.DataFrame(metric_rows)

if reconstruction_metrics.empty:
    print("No reconstruction/translation metrics could be computed.")
else:
    display(reconstruction_metrics.head())

    display(
        reconstruction_metrics
        .groupby(["split", "modality", "output_type"])[
            ["mse", "mae", "rmse", "p95_abs_error", "orig_std", "output_std"]
        ]
        .describe()
    )

    reconstruction_metrics.to_csv(
        report_dir / "reconstruction_translation_metrics_by_sample.csv",
        index=False,
    )


### Mean target original / reference / translation / error maps

For the configured target modality, this figure summarizes the average 3D structure across a subset of test patients. It allows the reference reconstruction and cross-modal translation to be inspected side-by-side against the original target volume.


In [ ]:
def compute_mean_target_outputs(
    result,
    *,
    target_modality,
    translation_key="translation",
    n_max=30,
):
    split_data = result.datasets.test
    dataset = split_data.datasets[target_modality]

    epoch = resolve_epoch(result.reconstructions, "test")
    recon_dict = result.reconstructions.get(epoch=epoch, split="test")
    id_dict = result.sample_ids.get(epoch=epoch, split="test")

    reference_key_local = f"reference_{target_modality}_to_{target_modality}"

    if not isinstance(recon_dict, dict):
        raise ValueError("Test reconstructions are not stored as a dictionary.")

    if translation_key not in recon_dict:
        raise KeyError("No cross-modal translation is stored.")

    if reference_key_local in recon_dict:
        reference_output_key = reference_key_local
    elif target_modality in recon_dict:
        reference_output_key = target_modality
    else:
        raise KeyError("No target-modality reference reconstruction is stored.")

    dataset_ids = np.asarray(dataset.sample_ids).astype(str)
    dataset_lookup = {
        sid: idx for idx, sid in enumerate(dataset_ids)
    }

    translation_ids = np.asarray(
        id_dict[translation_key]
    ).astype(str)
    reference_ids = np.asarray(
        id_dict[reference_output_key]
    ).astype(str)

    translation_lookup = {
        sid: idx for idx, sid in enumerate(translation_ids)
    }
    reference_lookup = {
        sid: idx for idx, sid in enumerate(reference_ids)
    }

    common_ids = [
        sid
        for sid in dataset_ids
        if sid in translation_lookup and sid in reference_lookup
    ][:n_max]

    if not common_ids:
        raise ValueError("No shared target/reference/translation sample IDs were found.")

    translation_values = to_numpy(recon_dict[translation_key])
    reference_values = to_numpy(recon_dict[reference_output_key])

    original_sum = None
    reference_sum = None
    translation_sum = None

    for sample_id in common_ids:
        original = extract_dataset_array(
            dataset,
            dataset_lookup[sample_id],
        )
        reference = np.asarray(
            reference_values[reference_lookup[sample_id]]
        ).squeeze().astype(np.float32)
        translation = np.asarray(
            translation_values[translation_lookup[sample_id]]
        ).squeeze().astype(np.float32)

        if original_sum is None:
            original_sum = np.zeros_like(original, dtype=np.float32)
            reference_sum = np.zeros_like(reference, dtype=np.float32)
            translation_sum = np.zeros_like(translation, dtype=np.float32)

        original_sum += original
        reference_sum += reference
        translation_sum += translation

    n = len(common_ids)

    return (
        original_sum / n,
        reference_sum / n,
        translation_sum / n,
        common_ids,
    )


try:
    mean_original, mean_reference, mean_translation, mean_map_ids = (
        compute_mean_target_outputs(
            result,
            target_modality=to_key,
            n_max=MEAN_MAP_MAX_SAMPLES,
        )
    )

    mean_translation_error = np.abs(
        mean_original - mean_translation
    )

    d_mid, h_mid, w_mid = [
        size // 2
        for size in mean_original.shape
    ]

    views = [
        (
            "Axial",
            lambda volume: volume[d_mid, :, :],
        ),
        (
            "Coronal",
            lambda volume: volume[:, h_mid, :],
        ),
        (
            "Sagittal",
            lambda volume: volume[:, :, w_mid],
        ),
    ]

    fig, axes = plt.subplots(
        3,
        4,
        figsize=(14, 11),
        constrained_layout=True,
    )

    intensity_values = np.concatenate(
        [
            mean_original.ravel(),
            mean_reference.ravel(),
            mean_translation.ravel(),
        ]
    )
    vmin, vmax = np.percentile(
        intensity_values,
        [1, 99],
    )

    for row, (view_name, slicer) in enumerate(views):
        panels = [
            ("Mean original", slicer(mean_original), "gray", vmin, vmax),
            ("Mean reference", slicer(mean_reference), "gray", vmin, vmax),
            ("Mean translation", slicer(mean_translation), "gray", vmin, vmax),
            (
                "Mean |translation error|",
                slicer(mean_translation_error),
                "magma",
                None,
                None,
            ),
        ]

        for col, (title, image, cmap, low, high) in enumerate(panels):
            axes[row, col].imshow(
                image,
                cmap=cmap,
                origin="lower",
                vmin=low,
                vmax=high,
            )
            axes[row, col].axis("off")

            if row == 0:
                axes[row, col].set_title(title)

            if col == 0:
                axes[row, col].set_ylabel(view_name)

    fig.suptitle(
        f"Mean target outputs: {from_key} -> {to_key} "
        f"(n={len(mean_map_ids)})",
        fontsize=14,
    )

    save_report_plot(
        fig,
        "mean_target_reference_translation_error",
        close=False,
    )
    plt.show()
    plt.close(fig)

except Exception as exc:
    print(f"Mean target translation maps unavailable: {exc}")


## Loss and latent-space visualizations

The inherited XModalix visualizations remain useful for XModalix3D because they operate on loss values and latent vectors rather than directly on 3D voxel arrays.

The 2D latent-space plot is qualitative: apparent overlap in a UMAP/PCA-style projection does **not** prove modality invariance. Quantitative alignment is evaluated separately below.


In [ ]:
# Only show loss plots that have non-zero values.

vis = xmodalix3d_loaded.visualizer

loss_df = vis._make_loss_format(
    result=result,
    config=xmodalix3d_loaded.config,
)

# Same XModalix filtering as visualize():
loss_df = loss_df[
    ~loss_df["Loss Term"].str.startswith(
        tuple(result.datasets.train.datasets.keys())
    )
]

# Remove loss terms that are entirely zero / NaN
valid_terms = [term
    for term in loss_df["Loss Term"].unique()
    if (loss_df.loc[loss_df["Loss Term"] == term, "Loss Value"].notna().any()
        and
        (loss_df.loc[loss_df["Loss Term"] == term, "Loss Value"] != 0).any()
    )
]

loss_df = loss_df[loss_df["Loss Term"].isin(valid_terms)].copy()
loss_df_absolute = loss_df[loss_df["Loss Term"] != "class_loss"].copy()
loss_df_relative = loss_df[loss_df["Loss Term"] != "clf_loss"].copy()

# Generate and cache both filtered loss plots explicitly.
# This avoids relying on stale plot objects created earlier in the pipeline.
vis.plots["loss_absolute"] = vis._make_loss_plot(
    df_plot=loss_df_absolute,
    plot_type="absolute",
)

vis.plots["loss_relative"] = vis._make_loss_plot(
    df_plot=loss_df_relative,
    plot_type="relative",
)

vis.show_loss(plot_type="absolute")
_ = save_report_plot(
        vis.plots["loss_absolute"],
        "loss_absolute",
        close=True,
    )

vis.show_loss(plot_type="relative")
_ = save_report_plot(
        vis.plots["loss_relative"],
        "loss_relative",
        close=True,
    )

### Modality-wise latent-space visualization

#### Valid (top) and test (bottom) scatter plot & valid ridgeline plot

In [ ]:
for split in ["valid", "test"]:
    
    epoch = resolve_epoch(result.latentspaces, split)

    xmodalix3d_loaded.visualizer.show_latent_space(
        result=result,
        plot_type="2D-scatter",
        epoch=epoch,
        split=split,
    )

    # There can be one plot per annotation parameter
    stored_plots = xmodalix3d_loaded.visualizer.plots["2D-scatter"][epoch][split]

    for param, plot_obj in stored_plots.items():
        save_report_plot(
            plot_obj,
            f"latent_2d_scatter_{split}_{param}",
            close=True,
        )


valid_epoch = resolve_epoch(result.latentspaces, "valid")

xmodalix3d_loaded.visualizer.show_latent_space(
    result=result,
    plot_type="Ridgeline",
    epoch=valid_epoch,
    split="valid",
)

stored_plots = xmodalix3d_loaded.visualizer.plots["Ridgeline"][valid_epoch]["valid"]

for param, plot_obj in stored_plots.items():
    save_report_plot(
        plot_obj,
        f"latent_ridgeline_valid_{param}",
        close=True,
    )

#### Scatter- & Ridgeline plot (all samples)

In [ ]:
xmodalix3d_loaded.visualizer.show_latent_space(
    result=result,
    plot_type="2D-scatter",
)

plot_obj = next(
    iter(
        xmodalix3d_loaded.visualizer.plots[
            "2D-scatter"
        ][-1]["all"].values()
    )
)
save_report_plot(
    plot_obj,
    "latent_2d_scatter_all",
    close=True,
)

xmodalix3d_loaded.visualizer.show_latent_space(
    result=result,
    plot_type="Ridgeline",
)

plot_obj = next(
    iter(
        xmodalix3d_loaded.visualizer.plots[
            "Ridgeline"
        ][-1]["all"].values()
    )
)
save_report_plot(
    plot_obj,
    "latent_ridgeline_all",
    close=True,
)

## Quantitative cross-modal alignment

The two XModalix3D-specific evaluation methods are used here.

### Modality linear probe

A logistic-regression classifier is trained on **frozen** embeddings to predict modality identity. Cross-validation is grouped by patient ID, so embeddings belonging to the same patient cannot be split between probe training and validation folds.

For $K$ modalities, balanced-accuracy chance level is $\frac{1}{K}$. Lower probe accuracy therefore indicates stronger modality invariance.

### Matched/unmatched analysis

For each modality pair, the mean latent distance between **matched measurements of the same patient** is compared with the mean distance between **different patients**:

$$
R = \frac{\overline d_{\mathrm{matched}}}{\overline d_{\mathrm{unmatched}}}
$$

Values below 1 indicate patient-specific cross-modal correspondence; smaller values indicate stronger correspondence.


In [ ]:
required_alignment_methods = [
    "modality_linear_probe",
    "matched_unmatched_analysis",
]

required_alignment_visualizations = [
    "show_modality_linear_probe",
    "show_matched_unmatched_analysis",
]

for method in required_alignment_methods:
    if not hasattr(xmodalix3d_loaded.evaluator, method):
        raise AttributeError(
            f"XModalix3DEvaluator is missing required method: {method}"
        )

for method in required_alignment_visualizations:
    if not hasattr(xmodalix3d_loaded.visualizer, method):
        raise AttributeError(
            f"XModal3DVisualizer is missing required method: {method}"
        )

print("Alignment evaluator and visualizer methods are available.")


### Final validation/test modality linear-probe results

In [ ]:
probe_summaries = []
probe_confusions = {}
representation = "z"

for split in ['valid','test']:
    
    store =  result.latentspaces
    epoch = resolve_epoch(store, split)

    summary_df, confusion_df = (
        xmodalix3d_loaded.evaluator.modality_linear_probe(
            result=result,
            representation=representation,
            split=split,
            epoch=epoch,
            n_splits=LINEAR_PROBE_CV_SPLITS,
            C=LINEAR_PROBE_C,
        )
    )

    summary_df = summary_df.copy()
    summary_df["display_epoch"] = (
            "final" if epoch == -1 else epoch + 1
    )

    probe_summaries.append(summary_df)
    probe_confusions[(split, representation)] = confusion_df

    confusion_df.to_csv(
        report_dir
        / f"modality_linear_probe_confusion_{split}_{representation}.csv"
    )

    xmodalix3d_loaded.visualizer.show_modality_linear_probe(
        summary_df=summary_df.drop(columns=["display_epoch"]),
        confusion_df=confusion_df,
    )

    probe_plot_obj = (
        xmodalix3d_loaded.visualizer.plots[
            "Modality-linear-probe"
        ][representation][split][epoch]
    )

    save_report_plot(
        probe_plot_obj,
        f"modality_linear_probe_{split}_{representation}",
        close=True,
    )

probe_summary_all = (
    pd.concat(probe_summaries, ignore_index=True)
    if probe_summaries
    else pd.DataFrame()
)

display(probe_summary_all)

if not probe_summary_all.empty:
    probe_summary_all.to_csv(
        report_dir / "modality_linear_probe_summary.csv",
        index=False,
    )

### Final validation/test matched/unmatched analysis

In [ ]:
alignment_summaries = []
alignment_distance_tables = {}
representation = "z"

for split in ['valid', 'test']:
    
    store =  result.latentspaces
    epoch = resolve_epoch(store, split)

    summary_df, distances_df = (
        xmodalix3d_loaded.evaluator.matched_unmatched_analysis(
            result=result,
            representation=representation,
            split=split,
            epoch=epoch,
            distance_metric="euclidean",
            return_distances=True,
        )
    )

    summary_df = summary_df.copy()
    summary_df["display_epoch"] = ("final" if epoch == -1 else epoch + 1)

    alignment_summaries.append(summary_df)
    alignment_distance_tables[(split, representation)] = distances_df

    summary_df.to_csv(
        report_dir
        / f"matched_unmatched_summary_{split}_{representation}.csv",
        index=False,
    )

    distances_df.to_csv(
        report_dir
        / f"matched_unmatched_distances_{split}_{representation}.csv",
        index=False,
    )

    xmodalix3d_loaded.visualizer.show_matched_unmatched_analysis(
        summary_df=summary_df.drop(columns=["display_epoch"]),
        distances_df=distances_df,
    )

    alignment_plot_obj = (
        xmodalix3d_loaded.visualizer.plots[
            "Matched-unmatched-alignment"
        ][representation][split][epoch]
    )

    save_report_plot(
        alignment_plot_obj,
        f"matched_unmatched_{split}_{representation}",
        close=True,
    )



alignment_summary_all = (
    pd.concat(alignment_summaries, ignore_index=True)
    if alignment_summaries
    else pd.DataFrame()
)

display(alignment_summary_all)

if not alignment_summary_all.empty:
    alignment_summary_all.to_csv(
        report_dir / "matched_unmatched_summary_all.csv",
        index=False,
    )

## Interpretation guide

For the two primary alignment diagnostics:

- **Modality linear probe:** lower is better for modality invariance. With three modalities, chance balanced accuracy is \(1/3 \approx 0.333\). High probe accuracy means modality identity remains linearly recoverable.
- **Matched/unmatched ratio:** lower is better for patient-specific correspondence. A ratio below 1 means matched patients are closer across modalities than unmatched patients.

These two properties should not be conflated. A model can preserve strong patient-specific correspondence while still retaining enough modality-specific structure for a linear classifier to identify the originating modality.

Unlike the Imagix3D report, this notebook does not currently include latent-activity or latent-traversal analyses because those methods belong to the separate `Imagix3DEvaluator` extension and have not yet been integrated into `XModalix3DEvaluator`.
